# Intuition test: LLMs and LoRA

Questions come first. For each one, write your prediction in the cell marked
`Prediction`, then run the code cell below it. A short answer follows each run.
Do not read the answer before you have committed to a prediction.

Covers notebooks 01 to 04. Seeded, offline, and self-contained. The model is a
two-layer GPT-2 built from scratch, so nothing is downloaded.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from llm_finetune import bpe, data, evaluate, model, train
from llm_finetune.config import TrainConfig

from common.dl import set_seed

set_seed(42)

## Q1. An unseen word and out-of-vocabulary

Train BPE on a tiny corpus that never contains the word `newer`. Then encode
`newer`. Will it fail with an out-of-vocabulary error, or still encode into
pieces? Write down which, and roughly how many pieces.

Prediction: *(write here)*

In [ ]:
corpus = ["low", "low", "lower", "newest", "newest", "widest"]
merges = bpe.train_bpe(corpus, num_merges=10)
pieces = bpe.encode("newer", merges)
print("newer ->", pieces)

Answer: it encodes, no error. BPE falls back to smaller subwords it learned
(`ne`, `wer`, plus the end marker). There is no out-of-vocabulary token because
every word decomposes down to single characters in the worst case.

## Q2. Subword length against character length

Take the string below. Split it on whitespace (word tokens) and split it into
characters. Which sequence is longer, and by roughly what factor? Where does a
BPE subword count sit relative to the two?

Prediction: *(write here)*

In [ ]:
text = "the newest widest harbour"
words = text.split()
chars = list(text)
subwords = bpe.encode("".join(text.split()), merges)
print("word tokens:", len(words))
print("subword tokens:", len(subwords))
print("char tokens:", len(chars))

Answer: characters are the longest sequence, several times longer than word
tokens. BPE sits between the two. Shorter sequences mean fewer positions to
attend over, which is why real models tokenise into subwords rather than
characters.

## Q3. LoRA output at initialisation

LoRA adds `B @ A` to a frozen weight, with `A` random and `B` initialised to
zero. Before any training step, is the LoRA layer's output the same as the base
layer's, or different? Write down which.

Prediction: *(write here)*

In [ ]:
import torch

set_seed(42)
base = torch.nn.Linear(16, 16, bias=False)
x = torch.randn(1, 16)

rank = 4
lora_a = torch.randn(rank, 16)  # random
lora_b = torch.zeros(16, rank)  # zero at init
update = x @ lora_a.T @ lora_b.T

base_out = base(x)
lora_out = base_out + update
print("max abs difference:", float((lora_out - base_out).abs().max()))

Answer: identical. `B = 0` makes `B @ A` exactly zero, so the update adds
nothing at init. Training moves `B` away from zero, and only then does the
adapter change the output. Starting from the base model's behaviour is what
makes LoRA safe to attach.

## Q4. How much becomes trainable

Attach LoRA (rank 8) to the tiny GPT-2 and count trainable parameters against
the total. Is it about half, about 10 per cent, or a few per cent?

Prediction: *(write here)*

In [ ]:
set_seed(42)
tiny_model, tokenizer = model.tiny_causal_lm()
config = TrainConfig(base_model="tiny", lora_rank=8, val_split=0.0)
lora_model = model.add_lora(tiny_model, config)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_model.parameters())
print(f"trainable: {trainable}")
print(f"total:     {total}")
print(f"fraction:  {100 * trainable / total:.2f}%")

Answer: a few per cent, about 2.7 on this tiny model. Only the small `A` and `B`
matrices on the attention projection carry gradients; the base weights stay
frozen. On a billion-parameter model the fraction is far smaller again, which is
the whole point of LoRA.

## Q5. Perplexity after a short fine-tune

Train the LoRA model for a few epochs on the demo corpus. Will perplexity on the
training texts go up or down? And will the generated text read as fluent
English? Write down both.

Prediction: *(write here)*

In [ ]:
set_seed(42)
train_config = TrainConfig(
    base_model="tiny",
    lora_rank=8,
    num_train_epochs=30,
    learning_rate=5e-3,
    val_split=0.0,
)
fresh_model, tokenizer = model.tiny_causal_lm()
lora_model = model.add_lora(fresh_model, train_config)

records = data.demo_corpus()
dataset = data.build_dataset(records, tokenizer, train_config)
texts = [data.format_instruct(record, tokenizer) for record in records]

before = evaluate.perplexity(lora_model, tokenizer, texts, "cpu")
train.train(lora_model, tokenizer, dataset, train_config)
after = evaluate.perplexity(lora_model, tokenizer, texts, "cpu")

print(f"perplexity before: {before:.2f}")
print(f"perplexity after:  {after:.2f}")

prompt = "Instruction:\nWhat is the capital of Marru?\n\nResponse:\n"
print("sample:", repr(evaluate.generate(lora_model, tokenizer, prompt, max_new_tokens=15)))

Answer: perplexity falls on the training data, here from about 511 to about 333.
The model is fitting the corpus. The generated text is still nonsense, because a
two-layer model with 150k parameters cannot produce fluent English. Lower
training perplexity and poor output are not a contradiction. The model is small
enough to memorise structure but too small to be fluent.

## Q6. Target modules by architecture

Does LoRA attach to the same module names on GPT-2 as it would on a Qwen or Llama
model? Write down yes or no, and the names if you know them.

Prediction: *(write here)*

In [ ]:
gpt2_targets = model.resolve_target_modules(tiny_model, config)
print("GPT-2 target modules:", gpt2_targets)
print("Qwen/Llama target modules: ['q_proj', 'v_proj']")

Answer: no. GPT-2 fuses the attention projection into one module called
`c_attn`, so LoRA attaches there. Qwen and Llama keep the query and value
projections separate, so LoRA attaches to `q_proj` and `v_proj`. The target
names follow the architecture, which is why `resolve_target_modules` inspects the
model rather than assuming one set of names.